# 06 · RAG 检索增强（真实 5.5 万条法条库 + 向量召回 + rerank 精排）

> 给 04 的 dpo_merged 外挂真实法条治幻觉。模型参数不变（in-context，不训练）。

## Pipeline（经实测定型）
`向量召回 top100 → bge-reranker 精排 → top3 → 拼进 prompt 生成`

## 数据
STARD 真实法条库 **55348 条**（github.com/oneal2000/STARD），向量已离线缓存 `data/stard_vecs.pt`。

## 实现要点与取舍
1. **玩具库假象**：10 条手写库检索"完美"是假象；换 5.5 万条真实库，纯向量检索的局限才暴露。
2. **BM25 实测拖后腿**：按最佳实践加了 BM25 混合检索，但中文法律高频词（"适用/规定/解释"）把 BM25 带偏，RRF 融合反而拖差了向量结果 → **最佳实践要用自己的数据实测，不能照搬**。
3. **召回 vs 排序**：诊断发现对口法条都在向量召回 top100 内（重婚罪258在第3、诉讼时效在22名），瓶颈在排序不在召回 → 对症的是 **rerank 精排**（cross-encoder），实测把重婚罪刑法258从第3顶到第1。
4. **RAG 效果 = 检索质量 × 模型利用检索的能力**：检索准则 RAG 强；检索错则模型"有依据地说错"，比幻觉更危险。

In [1]:
import os, json
import torch
import torch.nn.functional as F

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
import certifi; os.environ["SSL_CERT_FILE"] = certifi.where()

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
DATA = os.path.join(ROOT, "data")
OUT  = os.path.join(ROOT, "outputs")
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")

BGE_PATH      = os.path.join(ROOT, "models", "bge-small-zh-v1.5")
RERANKER_PATH = os.path.join(ROOT, "models", "bge-reranker-base")
DPO_MERGED    = os.path.join(OUT, "dpo_merged")
print("DEVICE:", DEVICE)

DEVICE: mps


In [2]:
# ---- 加载两个检索模型：bge(召回) + reranker(精排) ----
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

# 召回用 bi-encoder：文字→向量，快，适合在 5.5 万条里粗筛
emb_tok = AutoTokenizer.from_pretrained(BGE_PATH)
emb = AutoModel.from_pretrained(BGE_PATH).to(DEVICE).eval()
def embed(texts):
    enc = emb_tok(texts, padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = emb(**enc)
    return F.normalize(out.last_hidden_state[:, 0], p=2, dim=1)

# 精排用 cross-encoder：把(问题,法条)一起读，判相关性，准但慢，只排召回的候选
rer_tok = AutoTokenizer.from_pretrained(RERANKER_PATH)
reranker = AutoModelForSequenceClassification.from_pretrained(RERANKER_PATH).to(DEVICE).eval()
print("bge + reranker 就绪")

Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

bge + reranker 就绪


In [3]:
# ---- 读法条库 + 加载预编码向量（秒级，不重编）----
corpus = [json.loads(l) for l in open(os.path.join(DATA, "stard_corpus.jsonl"), encoding="utf-8") if l.strip()]
clause_vecs = torch.load(os.path.join(DATA, "stard_vecs.pt")).to(DEVICE)
print("法条数:", len(corpus), "| 向量矩阵:", tuple(clause_vecs.shape))

法条数: 55348 | 向量矩阵: (55348, 512)


In [4]:
# ---- 两段式检索：向量召回 top100 → reranker 精排 top3 ----
@torch.no_grad()
def retrieve(query, k=3, recall_n=100):
    # 1) 向量粗召回 recall_n 条（快，缩小范围）
    sims = (embed([query]) @ clause_vecs.T)[0]
    cand = sims.topk(recall_n).indices.tolist()
    # 2) reranker 对这 recall_n 条精排（把(问题,法条)拼一起打相关性分）
    pairs = [[query, corpus[i]["content"]] for i in cand]
    enc = rer_tok(pairs, padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
    scores = reranker(**enc).logits.view(-1)
    order = scores.argsort(descending=True).tolist()
    return [(corpus[cand[j]], float(scores[j])) for j in order[:k]]

# 测试：重婚罪应被 rerank 顶到刑法258
for q in ["盗窃罪判几年", "什么样算重婚罪"]:
    print("Q:", q)
    for c, s in retrieve(q):
        print(f"  {s:6.2f}  {c['name']}")
    print()

Q: 盗窃罪判几年
    6.28  中华人民共和国刑法第二百六十四条
    5.60  中华人民共和国刑法第一百二十七条
    5.46  中华人民共和国刑法第三百零二条

Q: 什么样算重婚罪
    2.24  中华人民共和国刑法第二百五十八条
    1.58  最高人民法院关于适用《刑事诉讼法》的解释第一条
    0.26  中华人民共和国民法典第一千零五十四条



In [5]:
# ---- 加载生成模型(04 的 dpo_merged) ----
from transformers import AutoModelForCausalLM
gen_tok = AutoTokenizer.from_pretrained(DPO_MERGED)
gen_model = AutoModelForCausalLM.from_pretrained(DPO_MERGED, torch_dtype=torch.float32).to(DEVICE)

def generate(system, user, max_new_tokens=256):
    msgs = [{"role":"system","content":system}, {"role":"user","content":user}]
    text = gen_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = gen_tok(text, return_tensors="pt").to(gen_model.device)
    out = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return gen_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("生成模型就绪")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

生成模型就绪


In [6]:
# ---- 无 RAG vs 有 RAG(召回+rerank) ----
SYSTEM = "你是专业的中文法律咨询助手。请针对用户的法律问题，给出准确、有依据、条理清晰的解答；涉及具体法条时尽量说明，不要编造不存在的法条或事实。"

def answer_no_rag(q):
    return generate(SYSTEM, q)

def answer_rag(q, k=3):
    hits = retrieve(q, k)
    ctx = "\n".join(f"{c['name']}：{c['content'].strip()}" for c, _ in hits)
    user = (f"请只依据下面提供的法条回答问题，不得编造法条编号或内容，并指出引用了哪一条。\n\n"
            f"【参考法条】\n{ctx}\n\n【问题】{q}")
    return hits, generate(SYSTEM, user)

In [7]:
# ---- demo：并排对比 ----
questions = ["什么样算重婚罪？", "盗窃罪一般判几年？", "婚前全款买的房子，离婚时要分给对方吗？"]
for q in questions:
    print("="*70); print("问题:", q); print("-"*70)
    print("【无 RAG】", answer_no_rag(q)[:220]); print("-"*70)
    hits, ans = answer_rag(q)
    print("检索到:", [c["name"] for c, _ in hits])
    print("【有 RAG】", ans[:300]); print()

问题: 什么样算重婚罪？
----------------------------------------------------------------------
【无 RAG】 根据《中华人民共和国刑法》第三百一十三条的规定，重婚是指有配偶者与他人结婚的行为。重婚是婚姻登记机关审查当事人是否符合结婚条件所必须具备的条件之一。重婚行为具有以下特征：1.重婚行为发生在夫妻关系存续期间；2.重婚行为发生在夫妻双方自愿的情况下；3.重婚行为是基于非法手段而形成的婚姻关系；4.重婚行为是违背社会公序良俗的违法行为。重婚罪在司法实践中表现为：1.重婚人与被重婚人的配偶共同生活；2.重婚人与被重婚人的子女共同生活；3.重婚人
----------------------------------------------------------------------
检索到: ['中华人民共和国刑法第二百五十八条', '中华人民共和国民法典第一千零五十四条', '最高人民法院关于适用《民法典》婚姻家庭编的解释（一）第九条']
【有 RAG】 重婚罪是指有配偶而重婚的，或者明知他人有配偶而与之结婚的行为。

问题: 盗窃罪一般判几年？
----------------------------------------------------------------------
【无 RAG】 盗窃罪在司法实践中，量刑从轻到重分为三年以下有期徒刑、拘役或者管制，并处罚金。根据最高人民法院《关于审理盗窃案件具体应用法律若干问题的解释》的规定，犯盗窃罪，有下列情形之一的，应当认定为刑法第二百六十四条规定的“数额较大”：（一）个人盗窃公私财物价值人民币200元至300元的；（二）个人盗窃公私财物价值人民币500元至1000元以上的；（三）入户盗窃或者携带凶器盗窃的；（四）在公共场所对群众实施盗窃的；（五）多次盗窃或者一年内盗窃两次以
----------------------------------------------------------------------
检索到: ['中华人民共和国刑法第二百六十四条', '中华人民共和国刑法第三百零二条', '中华人民共和国刑法第一百二十七条']
【有 RAG】 盗窃罪一般判十年以上有期徒刑。

问题: 婚前全款买的房子，离婚时要分给

## 改进方向
- embedding 换 bge-large 或法律领域微调（提升召回上限，让难 case 进 top100）。
- reranker 换 bge-reranker-v2-m3（更强）。
- 检索不到（rerank 最高分仍很低）就拒答；引用标注；上下文压缩。
- 评测：RAGAS（faithfulness / context recall / 引用正确性）。
- 进阶：retrieval-augmented SFT —— 用 (问题+检索法条)→答案 训练，让模型更会用检索结果。